На этом семинаре соберем пайплайн для offline-экспериментов с рекомендательными моделями на датасете MovieLens-1M:  

1. Посмотрим на датасет и сделаем k-core фильтрацию и честное
   train/val/test разбиение — глобальный temporal ratio-based сплит по
   реальным таймстемпам взаимодействий.
2. Реализуем офлайн-метрики: HitRate, Recall, NDCG, Coverage.
3. Построим бейзлайны — Random и TopPopular.
4. Обучим четыре модели: ItemKNN (Jaccard), ALS, EASE и
   SASRec (vanilla, BCE).
5. Сравним всё на едином лидерборде и посмотрим на персональные рекомендации.

## 0. Установка и импорты

In [ ]:
%pip install -q polars pandas numpy scipy scikit-learn matplotlib seaborn requests tqdm implicit torch

In [ ]:
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import requests
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from implicit.als import AlternatingLeastSquares
from IPython.display import display
from scipy.sparse import csr_matrix
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
DATA_DIR = Path("data/13_seminar")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(DEVICE)

## 1. Данные

[MovieLens-1M](https://grouplens.org/datasets/movielens/1m/) — классический
бенчмарк-датасет: явные рейтинги от 1 до 5, у каждого взаимодействия есть
таймстемп, это и позволяет сделать честный temporal split.

In [ ]:
def download(url: str, path: Path, chunk_size: int = 1 << 16) -> Path:
    """Скачивает файл по url в path, если его там ещё нет."""
    path = Path(path)
    if path.exists():
        return path
    path.parent.mkdir(parents=True, exist_ok=True)

    response = requests.get(url, stream=True, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    response.raise_for_status()
    total = int(response.headers.get("content-length", 0))

    tmp_path = path.with_suffix(path.suffix + ".part")
    with open(tmp_path, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=path.name, leave=False) as pbar:
        for chunk in response.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))
    tmp_path.rename(path)
    return path


def download_and_unzip(url: str, extract_root: Path, name: str) -> Path:
    """Скачивает zip-архив и один раз распаковывает его в extract_root/name."""
    extract_root = Path(extract_root)
    marker = extract_root / f".{name}.ready"
    if not marker.exists():
        extract_root.mkdir(parents=True, exist_ok=True)
        archive = download(url, extract_root / f"{name}.zip")
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(extract_root)
        marker.touch()
    return extract_root / name

In [ ]:
ML1M_DIR = download_and_unzip(
    "https://files.grouplens.org/datasets/movielens/ml-1m.zip", DATA_DIR, "ml-1m",
)

ml_ratings = pd.read_csv(
    ML1M_DIR / "ratings.dat", sep="::", engine="python",
    names=["user_id", "item_id", "rating", "timestamp"],
)
ml_movies = pd.read_csv(
    ML1M_DIR / "movies.dat", sep="::", engine="python", encoding="latin-1",
    names=["item_id", "title", "genres"],
)

ml_ratings = pl.from_pandas(ml_ratings)
ml_movies = pl.from_pandas(ml_movies)

ml_movies = ml_movies.with_columns(
    pl.col("title").str.extract(r"\((\d{4})\)\s*$", 1).cast(pl.Int32).alias("year"),
)

print(
    f"MovieLens-1M: {ml_ratings.height} оценок, "
    f"{ml_ratings['user_id'].n_unique()} пользователей, "
    f"{ml_ratings['item_id'].n_unique()} фильмов"
)
ml_movies.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x=ml_ratings["rating"].to_numpy(), ax=axes[0])
axes[0].set(title="Распределение оценок", xlabel="Оценка", ylabel="Количество")

popularity = (
    ml_ratings["item_id"].value_counts(normalize=True, name="proportion")
    .sort("proportion", descending=True)["proportion"]
    .to_numpy()
)
axes[1].plot(np.arange(1, len(popularity) + 1), np.cumsum(popularity))
axes[1].set(
    title="Popularity bias",
    xlabel="Фильмы, отсортированные по популярности", ylabel="Накопленная доля оценок",
)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

genre_counts = (
    ml_movies["genres"].str.split("|").explode().value_counts()
    .sort("count", descending=True).head(12)
)
axes[0].barh(genre_counts["genres"].to_numpy()[::-1], genre_counts["count"].to_numpy()[::-1])
axes[0].set(title="Число фильмов по жанрам", xlabel="Фильмы")

dates = pd.to_datetime(ml_ratings["timestamp"].to_numpy(), unit="s")
pd.Series(1, index=dates).resample("ME").sum().plot(ax=axes[1])
axes[1].set(title="Оценки по месяцам", xlabel="Месяц", ylabel="Оценки")

plt.tight_layout()
plt.show()

In [ ]:
topn = 10
ml_top_popular = ... # TODO: найти топ фильмов по кол-ву рейтингов, отсортировать по убыванию популярности

assert ml_top_popular.height == topn
ml_top_popular.select("item_id", "title", "n_ratings")

## 2. Предобработка: k-core фильтрация и train/val/test сплит


### k-core фильтрация

Модели вроде kNN или ALS плохо работают на пользователях/айтемах с единичными
взаимодействиями — по ним просто нет сигнала. Стандартный приём — k-core
фильтрация: оставить только пользователей и айтемы, у которых не меньше
`k_core` взаимодействий.

Важный нюанс: фильтрация в один проход не гарантирует, что после неё каждый
пользователь и каждый айтем всё ещё удовлетворяют порогу — удалив "холодные"
айтемы, мы можем опустить каких-то пользователей ниже `k_core`, и наоборот.
Поэтому делаем это итеративно, пока размер датасета не перестанет
меняться.

In [ ]:
def iterative_k_core(
    df: pl.DataFrame,
    k_core: int,
    user_col: str = "user_id",
    item_col: str = "item_id",
    max_iters: int = 100,
) -> pl.DataFrame:
    """Итеративная k-core фильтрация: оставляем пользователей и айтемы с
    >= k_core взаимодействиями, повторяя до сходимости."""
    prev_size = -1
    for _ in range(max_iters):
        user_counts = df.group_by(user_col).len()
        valid_users = ... # TODO

        item_counts = df.group_by(item_col).len()
        valid_items = ... # TODO

        df = df.filter(pl.col(user_col).is_in(valid_users) & pl.col(item_col).is_in(valid_items))

        if df.height == prev_size:
            break
        prev_size = df.height
    return df

### Train / validation / test сплит

Для рекомендательных систем случайное разбиение (когда для
каждого пользователя случайно откладывается часть его истории в тест) на
практике опасен: он неявно даёт модели заглянуть в будущее — например,
взаимодействие из 2001 года может оказаться в train, а взаимодействие того же
пользователя из 2000 года — в test, и по факту модель обучается "предсказывать
прошлое по будущему".

Более честный протокол — global temporal split: сортируем все
взаимодействия (по всем пользователям сразу) по времени и режем единую
временную шкалу по глобальным квантилям на train / val / test. Так test
гарантированно целиком позже train и val, и мы валидируем модель в реалистичном
сценарии "обучились на истории — предсказываем будущее". "Ratio-based" здесь
означает, что границы сплита выбираются не по фиксированной дате, а по доле
данных (например, последние 10% интеракций по времени — в test), что делает сплит
устойчивым к разной плотности логов в разные периоды.

In [ ]:
def global_temporal_split(
    df: pl.DataFrame,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    timestamp_col: str = "timestamp",
) -> Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """Глобальный temporal ratio-based сплит: режем всю историю взаимодействий
    по времени на train / val / test так, чтобы test был строго позже val,
    а val — строго позже train."""
    assert 0 < val_ratio < 1 and 0 < test_ratio < 1 and val_ratio + test_ratio < 1

    train_cutoff, val_cutoff = df.select(
        pl.col(timestamp_col).quantile(1 - val_ratio - test_ratio).alias("train_cutoff"),
        pl.col(timestamp_col).quantile(1 - test_ratio).alias("val_cutoff"),
    ).row(0)

    train = df.filter(pl.col(timestamp_col) <= train_cutoff)
    val = df.filter((pl.col(timestamp_col) > train_cutoff) & (pl.col(timestamp_col) <= val_cutoff))
    test = df.filter(pl.col(timestamp_col) > val_cutoff)
    return train, val, test


def restrict_to_known(eval_df: pl.DataFrame, train_df: pl.DataFrame, user_col="user_id", item_col="item_id") -> pl.DataFrame:
    """Оставляет в val/test только warm-start пары: пользователь и айтем уже
    встречались в train. Иначе метрики штрафовали бы модель за пользователей и
    айтемы, о которых она принципиально не могла ничего узнать."""
    return eval_df.filter(
        pl.col(user_col).is_in(train_df[user_col].unique().to_list())
        & pl.col(item_col).is_in(train_df[item_col].unique().to_list())
    )


def ground_truth_dict(df: pl.DataFrame, user_col: str = "user_id", item_col: str = "item_id") -> Dict:
    """train/val/test в формат user_id -> set(item_id), где ground truth —
    все айтемы пользователя в этом наборе."""
    grouped = df.group_by(user_col).agg(pl.col(item_col))
    return {row[0]: set(row[1]) for row in grouped.iter_rows()}


def prepare_dataset(
    ratings: pl.DataFrame,
    k_core: int,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    timestamp_col: str = "timestamp",
) -> Dict:
    """Единый пайплайн предобработки: k-core фильтрация -> temporal сплит ->
    warm-start фильтрация val/test -> словари ground truth для метрик."""
    filtered = iterative_k_core(ratings, k_core=k_core)
    train, val, test = global_temporal_split(filtered, val_ratio, test_ratio, timestamp_col)

    val = restrict_to_known(val, train)
    test = restrict_to_known(test, train)

    return {
        "train": train,
        "val": val,
        "test": test,
        "train_val": pl.concat([train, val]),
        "val_truth": ground_truth_dict(val),
        "test_truth": ground_truth_dict(test),
        "catalog": train["item_id"].unique().to_list(),
        "timestamp_col": timestamp_col,
    }

In [ ]:
ML_K_CORE = 5
ML_VAL_RATIO, ML_TEST_RATIO = 0.1, 0.1

ml_data = prepare_dataset(
    ml_ratings, k_core=ML_K_CORE,
    val_ratio=ML_VAL_RATIO, test_ratio=ML_TEST_RATIO,
    timestamp_col="timestamp",
)

print(
    f"MovieLens после k-core={ML_K_CORE} и temporal split: "
    f"train={ml_data['train'].height}, val={ml_data['val'].height}, test={ml_data['test'].height}"
)
print(
    f"train: {ml_data['train']['user_id'].n_unique()} пользователей, {ml_data['train']['item_id'].n_unique()} фильмов\n"
    f"val: {ml_data['val']['user_id'].n_unique()} пользователей, {ml_data['val']['item_id'].n_unique()} фильмов\n"
    f"test: {ml_data['test']['user_id'].n_unique()} пользователей, {ml_data['test']['item_id'].n_unique()} фильмов"
)

## 3. Метрики

Нотация:
- $k$ — глубина отсечки (первые $k$ рекомендаций);
- $S_u^{(k)} = [i_1, \dots, i_k]$ — упорядоченный список рекомендаций пользователю $u$;
- $T_u$ — множество релевантных (отложенных в val/test) айтемов пользователя $u$;
- $\mathbb{I}[\cdot]$ — индикаторная функция.

Пользователи без единого релевантного айтема ($T_u = \emptyset$) не участвуют
в усреднении — метрика для них попросту не определена.

**HitRate@k** — попал ли хотя бы один релевантный айтем в топ-k:

$$\text{HitRate}(u, k) = \mathbb{I}\left[S_u^{(k)} \cap T_u \neq \emptyset\right]$$

**Recall@k** — какая доля релевантных айтемов пользователя оказалась в топ-k:

$$\text{Recall}(u, k) = \frac{|S_u^{(k)} \cap T_u|}{|T_u|}$$

**NDCG@k** — учитывает не только наличие релевантных айтемов, но и их позицию
(чем выше — тем ценнее), нормированная на идеальный порядок выдачи:

$$
\text{DCG}(u, k) = \sum_{j=1}^{k} \frac{\mathbb{I}[i_j \in T_u]}{\log_2(j + 1)},
\qquad
\text{NDCG}(u, k) = \frac{\text{DCG}(u, k)}{\text{IDCG}(u, k)}
$$

где $\text{IDCG}(u, k)$ — DCG при идеальном порядке (все релевантные айтемы
идут первыми).

**Coverage@k** — метрика разнообразия каталога, а не качества для конкретного
пользователя: какая доля всего каталога вообще хоть раз встретилась в топ-k
хоть у одного пользователя. Модель, штампующая всем одну и ту же "безопасную"
рекомендацию, будет иметь низкий Coverage:

$$\text{Coverage}(k) = \frac{\left|\bigcup_{u \in U} S_u^{(k)}\right|}{|I|}$$

In [ ]:
def _binary_relevance_dcg(hits: np.ndarray) -> float:
    ranks = np.arange(1, len(hits) + 1)
    return float(np.sum(hits / np.log2(ranks + 1)))


def hit_rate_at_k(recommendations: Dict, ground_truth: Dict, k: int = 10) -> float:
    """Доля пользователей, у которых среди первых k рекомендаций встретился
    хотя бы один релевантный айтем."""
    hits = []
    for user_id, truth in ground_truth.items():
        if not truth:
            continue
        ... # TODO
    return float(np.mean(hits)) if hits else 0.0


def recall_at_k(recommendations: Dict, ground_truth: Dict, k: int = 10) -> float:
    """Средняя по пользователям доля релевантных айтемов, попавших в топ-k."""
    values = []
    for user_id, truth in ground_truth.items():
        if not truth:
            continue
        ... # TODO
    return float(np.mean(values)) if values else 0.0


def ndcg_at_k(recommendations: Dict, ground_truth: Dict, k: int = 10) -> float:
    """Normalized Discounted Cumulative Gain с бинарной релевантностью."""
    values = []
    for user_id, truth in ground_truth.items():
        if not truth:
            continue
        recs = recommendations.get(user_id, [])[:k]
        gains = np.array([1.0 if item in truth else 0.0 for item in recs])
        ... # TODO
    return float(np.mean(values)) if values else 0.0


def coverage_at_k(recommendations: Dict, catalog: Sequence, k: int = 10) -> float:
    """Доля каталога, покрытая объединением top-k рекомендаций по всем пользователям."""
    recommended = set()
    for recs in recommendations.values():
        ... # TODO
    return len(recommended) / len(catalog) if len(catalog) else 0.0

In [ ]:
def evaluate_model(
    model,
    ground_truth: Dict,
    catalog: Sequence,
    ks: Sequence[int] = (5, 10, 20),
    model_name: str = "model",
) -> pl.DataFrame:
    """Считает HitRate/Recall/NDCG/Coverage@k для набора k одним проходом
    (рекомендации строятся один раз с topn = max(ks))."""
    topn = max(ks)
    recommendations = model.recommend_to_users(list(ground_truth.keys()), topn=topn)

    rows = []
    for k in ks:
        rows.append({
            "model": model_name,
            "k": k,
            "HitRate": hit_rate_at_k(recommendations, ground_truth, k),
            "Recall": recall_at_k(recommendations, ground_truth, k),
            "NDCG": ndcg_at_k(recommendations, ground_truth, k),
            "Coverage": coverage_at_k(recommendations, catalog, k),
        })
    return pl.DataFrame(rows)

Ниже везде, где мы смотрим на качество — и бейзлайны, и каждая модель по
отдельности на `val`, и финальное сравнение на `test`, — будем вызывать
`evaluate_model`, передавая ей `ml_data["val_truth"]` (или `ml_data["test_truth"]`
для финального сравнения) как ground truth.


## 4. Базовый класс рекомендера

Все модели ниже наследуются от одного базового класса: он строит разреженную
матрицу взаимодействий `interactions_matrix` (пользователи × айтемы) и словари
для перехода между исходными id и их плотными индексами в матрице.

In [ ]:
class BaseRecommender:
    def __init__(
        self,
        interactions: pl.DataFrame,
        user_col: str = "user_id",
        item_col: str = "item_id",
        value_col: Optional[str] = "rating",
    ):
        self.user_col = user_col
        self.item_col = item_col

        self.id2user = interactions[user_col].unique().to_numpy()
        self.id2item = interactions[item_col].unique().to_numpy()
        self.user2id = {u: idx for idx, u in enumerate(self.id2user)}
        self.item2id = {i: idx for idx, i in enumerate(self.id2item)}

        rows = np.array([self.user2id[u] for u in interactions[user_col]], dtype=np.int64)
        cols = np.array([self.item2id[i] for i in interactions[item_col]], dtype=np.int64)

        self.n_users, self.n_items = len(self.id2user), len(self.id2item)

        # Бинарная матрица "было ли взаимодействие вообще" — на ней считаются
        # ItemKNN, EASE и вообще всё, для чего важен факт интереса, а не его сила.
        binary_data = np.ones(len(rows), dtype=np.float32)
        self.interactions_matrix = csr_matrix((binary_data, (rows, cols)), shape=(self.n_users, self.n_items))

        # Матрица с исходными значениями (рейтинг явный, либо 1 — если явной оценки нет) —
        # нужна там, где сила сигнала имеет значение (ALS).
        if value_col is not None and value_col in interactions.columns:
            value_data = interactions[value_col].to_numpy().astype(np.float32)
        else:
            value_data = binary_data
        self.ratings_matrix = csr_matrix((value_data, (rows, cols)), shape=(self.n_users, self.n_items))

    def _get_user_items(self, user_idx: int) -> set:
        return set(self.interactions_matrix[user_idx].indices)

    def _rank_scores(self, scores: np.ndarray, user_indices: np.ndarray, topn: int, exclude_seen: bool = True) -> np.ndarray:
        """Быстрый top-n по плотной матрице скоров: обнуляет уже увиденные
        айтемы, затем находит top-n через argpartition (без полной сортировки
        всего каталога) и лишь потом сортирует сами top-n."""
        scores = np.array(scores, dtype=np.float32, copy=True)
        if exclude_seen:
            seen_mask = self.interactions_matrix[user_indices].toarray().astype(bool)
            scores[seen_mask] = -np.inf

        topn = min(topn, scores.shape[1])
        top_idx = np.argpartition(-scores, topn - 1, axis=1)[:, :topn]
        rows = np.arange(scores.shape[0])[:, None]
        order = np.argsort(-scores[rows, top_idx], axis=1)
        return top_idx[rows, order]

    def recommend_to_users(self, user_ids: Sequence, topn: int = 10) -> Dict:
        raise NotImplementedError

## 5. Бейзлайны

Прежде чем сравнивать сложные модели, нужна точка отсчёта:

- **Random** — рекомендует случайные ещё не увиденные айтемы. Любая модель,
  которая не обыгрывает Random, бесполезна.
- **TopPopular** — рекомендует всем одни и те же самые популярные айтемы (не
  считая уже увиденных). Обыграть TopPopular по Recall/NDCG сложнее, чем
  кажется — популярность является сильным сигналом, но у неё почти нулевой
  Coverage и никакой персонализации.

In [ ]:
class RandomRecommender(BaseRecommender):
    def __init__(self, interactions: pl.DataFrame, seed: int = 42, **kwargs):
        super().__init__(interactions, **kwargs)
        self.rng = np.random.default_rng(seed)
        self._all_items = np.arange(self.n_items)

    def recommend_to_users(self, user_ids: Sequence, topn: int = 10) -> Dict:
        recommendations = {}
        for uid in user_ids:
            seen = self._get_user_items(self.user2id[uid]) if uid in self.user2id else set()
            candidates = np.array([idx for idx in self._all_items if idx not in seen])
            size = min(topn, len(candidates))
            chosen = self.rng.choice(candidates, size=size, replace=False) if size else np.array([], dtype=np.int64)
            recommendations[uid] = self.id2item[chosen].tolist()
        return recommendations


class TopPopularRecommender(BaseRecommender):
    def __init__(self, interactions: pl.DataFrame, **kwargs):
        super().__init__(interactions, **kwargs)
        popularity = np.asarray(self.interactions_matrix.sum(axis=0)).ravel()
        self.ranked_items = np.argsort(-popularity)

    def recommend_to_users(self, user_ids: Sequence, topn: int = 10) -> Dict:
        recommendations = {}
        for uid in user_ids:
            seen = self._get_user_items(self.user2id[uid]) if uid in self.user2id else set()
            top = [idx for idx in self.ranked_items if idx not in seen][:topn]
            recommendations[uid] = self.id2item[top].tolist()
        return recommendations

In [ ]:
random_model = RandomRecommender(ml_data["train"])
evaluate_model(random_model, ml_data["val_truth"], ml_data["catalog"], model_name="Random")

In [ ]:
top_popular_model = TopPopularRecommender(ml_data["train"])
evaluate_model(top_popular_model, ml_data["val_truth"], ml_data["catalog"], model_name="TopPopular")

## 6. ItemKNN (Jaccard)

Классический item-based коллаборативный kNN: два айтема похожи, если их
смотрели/оценивали примерно одни и те же пользователи. Похожесть — индекс
Жаккара по бинарным векторам взаимодействий:

$$J(i, j) = \frac{|U_i \cap U_j|}{|U_i \cup U_j|}$$

где $U_i$ — множество пользователей, взаимодействовавших с айтемом $i$.

Чтобы не считать и не хранить плотную матрицу $|I| \times |I|$, действуем так:

1. Пересечения $|U_i \cap U_j|$ для всех пар — это просто `X^T @ X` (матрица
   со-встречаемости по бинарной матрице интеракций), причём результат уже
   разрежен: ненулевые элементы есть только у пар айтемов с общими
   пользователями.
2. Объединения довосстанавливаем по формуле $|U_i| + |U_j| - |U_i \cap U_j|$.
3. Для каждого айтема оставляем только top-`k_neighbours` самых похожих соседей
   — так матрица схожести остаётся разреженной и скоринг пользователей
   считается одним sparse matmul.

Скор айтема для пользователя — сумма схожести с айтемами из его истории:

$$\text{score}(u, j) = \sum_{i \in U_u} J(i, j)$$

то есть матрично: `scores = interactions_matrix[users] @ similarity_matrix`.

In [ ]:
class ItemKNNRecommender(BaseRecommender):
    def __init__(self, interactions: pl.DataFrame, k_neighbours: int = 100, alpha: float = 1e-3, **kwargs):
        super().__init__(interactions, **kwargs)
        self.k_neighbours = k_neighbours
        self.alpha = alpha
        self.similarity = self._build_similarity()

    def _build_similarity(self) -> csr_matrix:
        item_counts = np.asarray(self.interactions_matrix.sum(axis=0)).ravel()
        cooccurrence = ... # TODO

        rows, cols, values = [], [], []
        for item_idx in range(self.n_items):
            start, end = cooccurrence.indptr[item_idx], cooccurrence.indptr[item_idx + 1]
            neighbour_idx = cooccurrence.indices[start:end]
            intersections = cooccurrence.data[start:end]

            unions = item_counts[item_idx] + item_counts[neighbour_idx] - intersections
            similarities = intersections / (unions + self.alpha)

            self_mask = neighbour_idx == item_idx
            similarities = similarities.copy()
            similarities[self_mask] = -1.0

            k = min(self.k_neighbours, len(similarities))
            if k <= 0:
                continue
            top = np.argpartition(-similarities, k - 1)[:k]
            top = top[similarities[top] > 0]

            rows.extend([item_idx] * len(top))
            cols.extend(neighbour_idx[top])
            values.extend(similarities[top])

        return csr_matrix((values, (rows, cols)), shape=(self.n_items, self.n_items))

    def recommend_to_users(self, user_ids: Sequence, topn: int = 10) -> Dict:
        valid = [uid for uid in user_ids if uid in self.user2id]
        recommendations = {uid: [] for uid in user_ids}
        if valid:
            user_indices = np.array([self.user2id[u] for u in valid], dtype=np.int64)
            scores = ... # TODO
            top_idx = self._rank_scores(scores, user_indices, topn)
            for uid, row in zip(valid, top_idx):
                recommendations[uid] = self.id2item[row].tolist()
        return recommendations

In [ ]:
itemknn_model = ItemKNNRecommender(ml_data["train"], k_neighbours=100)
evaluate_model(itemknn_model, ml_data["val_truth"], ml_data["catalog"], model_name="ItemKNN")

## 7. ALS

Alternating Least Squares для implicit-фидбэка (Hu, Koren, Volinsky, 2008):
матрица взаимодействий приближается произведением низкоранговых факторов
пользователей и айтемов $\hat{r}_{ui} = p_u^\top q_i$, но вместо самого
рейтинга оптимизируется confidence-взвешенная реконструкция бинарного
факта взаимодействия:

$$c_{ui} = 1 + \alpha \cdot r_{ui}$$

Чем больше взаимодействий (или выше рейтинг) — тем увереннее модель в том, что
пара действительно нравится пользователю, и тем сильнее штраф за ошибку на этой
паре. Обучение чередует (отсюда "alternating") решение системы уравнений
сначала по пользовательским факторам при фиксированных айтемных, потом
наоборот — каждый шаг имеет аналитическое решение (least squares), что и
делает метод быстрым по сравнению с SGD-подходами.

Используем реализацию из библиотеки `implicit`.

In [ ]:
class ALSRecommender(BaseRecommender):
    def __init__(
        self,
        interactions: pl.DataFrame,
        factors: int = 64,
        iterations: int = 15,
        regularization: float = 0.05,
        alpha: float = 20.0,
        seed: int = 42,
        **kwargs,
    ):
        super().__init__(interactions, **kwargs)

        # c_ui = 1 + alpha * r_ui (Hu et al., 2008). Там, где рейтинга нет
        # (или он "неинформативный"), считаем силу сигнала минимальной
        # положительной (1), а не отрицательной или нулевой.
        confidence = self.ratings_matrix.copy()
        confidence.data = 1.0 + alpha * np.clip(confidence.data, 1.0, None)

        self.model = AlternatingLeastSquares(
            factors=factors, iterations=iterations, regularization=regularization, random_state=seed,
        )
        self.model.fit(confidence)
        self.user_factors = self.model.user_factors
        self.item_factors = self.model.item_factors

    def recommend_to_users(self, user_ids: Sequence, topn: int = 10) -> Dict:
        valid = [uid for uid in user_ids if uid in self.user2id]
        recommendations = {uid: [] for uid in user_ids}
        if valid:
            user_indices = np.array([self.user2id[u] for u in valid], dtype=np.int64)
            scores = ... # TODO
            top_idx = self._rank_scores(scores, user_indices, topn)
            for uid, row in zip(valid, top_idx):
                recommendations[uid] = self.id2item[row].tolist()
        return recommendations

In [ ]:
als_model = ALSRecommender(ml_data["train"], factors=64, iterations=15, regularization=0.05, alpha=20.0)
evaluate_model(als_model, ml_data["val_truth"], ml_data["catalog"], model_name="ALS")

## 8. EASE (Embarrassingly Shallow Autoencoder)

EASE (Steck, 2019) — ещё одна item-item модель, но с не итеративным, а с closed-form решением. Идея: выучить матрицу весов $B$
($|I| \times |I|$, $\mathrm{diag}(B) = 0$), такую что $X B \approx X$ —
реконструируем матрицу взаимодействий как линейную комбинацию **других**
айтемов пользователя, запрещая айтему объяснять сам себя (иначе тривиальное
решение $B = I$):

$$\min_B \|X - XB\|_F^2 + \lambda\|B\|_F^2, \quad \mathrm{diag}(B) = 0$$

У этой задачи (если снять только ограничение на диагональ) есть аналитическое
решение через матрицу Грама $G = X^\top X + \lambda I$:

$$P = G^{-1}, \qquad B_{ij} = \begin{cases} -P_{ij} / P_{jj}, & i \neq j \\ 0, & i = j \end{cases}$$

То есть вся модель — это одна инверсия матрицы $|I| \times |I|$, без
градиентного спуска и подбора learning rate. На масштабе ml-1m (несколько
тысяч фильмов) это считается за секунды; для каталогов с сотнями тысяч
айтемов такая инверсия быстро становится главным узким местом метода
($O(|I|^3)$).

In [ ]:
class EASERecommender(BaseRecommender):
    def __init__(self, interactions: pl.DataFrame, reg: float = 100.0, **kwargs):
        super().__init__(interactions, **kwargs)

        X = self.interactions_matrix
        ... # TODO
        self.item_weights = weights.astype(np.float32)

    def recommend_to_users(self, user_ids: Sequence, topn: int = 10) -> Dict:
        valid = [uid for uid in user_ids if uid in self.user2id]
        recommendations = {uid: [] for uid in user_ids}
        if valid:
            user_indices = np.array([self.user2id[u] for u in valid], dtype=np.int64)
            scores = ... # TODO
            top_idx = self._rank_scores(scores, user_indices, topn)
            for uid, row in zip(valid, top_idx):
                recommendations[uid] = self.id2item[row].tolist()
        return recommendations

In [ ]:
ease_model = EASERecommender(ml_data["train"], reg=100.0)
evaluate_model(ease_model, ml_data["val_truth"], ml_data["catalog"], model_name="EASE")

## 9. SASRec

Все модели выше смотрят на историю пользователя как на множество
взаимодействий — им всё равно, в каком порядке пользователь смотрел фильмы.
SASRec (Kang, McAuley, 2018) моделирует историю как последовательность:
следующий айтем предсказывается по предыдущим через self-attention с каузальным маскированием — как в GPT, только вместо токенов текста у
нас айтемы каталога.

**Обучение.** Для последовательности $[i_1, \dots, i_n]$ на каждой позиции
$t$ модель по префиксу $i_1, \dots, i_t$ должна отличить настоящий следующий
айтем $i_{t+1}$ (позитив) от случайно засэмплированного "постороннего" айтема
(негатив) — по одному негативу на позицию, как в оригинальной статье. Loss —
обычная бинарная кросс-энтропия по скору $\langle h_t, e_i \rangle$
(скалярное произведение скрытого состояния и эмбеддинга айтема):

$$\mathcal{L} = -\sum_{u} \sum_{t} \Big[\log \sigma(\langle h_t, e_{i_{t+1}}\rangle) + \log\big(1 - \sigma(\langle h_t, e_{i^-}\rangle)\big)\Big]$$

**Как выбирать негатив $i^-$.** Сэмплируем его равномерно случайно из всего
каталога — простейший и стандартный вариант.

In [ ]:
def build_user_sequences(df: pl.DataFrame, item2id: Dict, timestamp_col: str = "timestamp") -> Dict:
    """user_id -> список индексов айтемов (со сдвигом +1, 0 зарезервирован под
    паддинг), отсортированный по времени взаимодействия."""
    ordered = df.sort(timestamp_col)
    grouped = ordered.group_by("user_id", maintain_order=True).agg(pl.col("item_id"))
    return {
        row[0]: [item2id[item] + 1 for item in row[1] if item in item2id]
        for row in grouped.iter_rows()
    }

In [ ]:
@dataclass
class SASRecConfig:
    max_len: int = 120
    hidden_dim: int = 64
    n_heads: int = 2
    n_blocks: int = 2
    dropout: float = 0.2
    n_epochs: int = 100
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 0.0
    grad_clip: float = 5.0
    eval_every: int = 10
    patience: int = 3  # сколько eval-проверок подряд без улучшения ждём перед остановкой
    inference_batch_size: int = 512


class SASRecDataset(Dataset):
    """Готовит для каждого пользователя один обучающий пример: паддированную
    слева последовательность входов, целевых позитивов, одного засэмплированного
    негатива на позицию и маску реальных (не паддинговых) позиций.

    Негатив сэмплируется равномерно по всему каталогу — без ограничения
    каким-либо пулом и без отбрасывания коллизий с реальной историей
    пользователя."""

    def __init__(
        self,
        sequences: Dict[int, List[int]],
        n_items: int,
        max_len: int,
        seed: int = 42,
    ):
        self.user_ids = [u for u, seq in sequences.items() if len(seq) >= 2]
        self.sequences = sequences
        self.n_items = n_items
        self.max_len = max_len
        self.rng = np.random.default_rng(seed)

    def __len__(self) -> int:
        return len(self.user_ids)

    def __getitem__(self, idx: int):
        seq = self.sequences[self.user_ids[idx]][-(self.max_len + 1):]

        input_seq = ... # TODO
        target_seq = ... # TODO
        negatives = ... # TODO

        pad_len = self.max_len - len(input_seq)
        input_seq = [0] * pad_len + input_seq
        target_seq = [0] * pad_len + target_seq
        negatives = [0] * pad_len + negatives
        mask = [0.0] * pad_len + [1.0] * len(target_seq[pad_len:])

        return (
            torch.tensor(input_seq, dtype=torch.long),
            torch.tensor(target_seq, dtype=torch.long),
            torch.tensor(negatives, dtype=torch.long),
            torch.tensor(mask, dtype=torch.float32),
        )

In [ ]:
class SASRec(nn.Module):
    def __init__(self, n_items: int, max_len: int, hidden_dim: int = 64, n_heads: int = 2, n_blocks: int = 2, dropout: float = 0.0):
        super().__init__()
        self.n_items = n_items
        self.max_len = max_len
        self.hidden_dim = hidden_dim

        # index 0 переиспользуется и под padding, и под "нет айтема" — его
        # эмбеддинг зафиксирован в нуле (padding_idx) и никогда не участвует
        # в предсказаниях (см. score_all_items).
        self.item_embedding = nn.Embedding(n_items + 1, hidden_dim, padding_idx=0)
        self.position_embedding = nn.Embedding(max_len, hidden_dim)
        self.embedding_dropout = nn.Dropout(dropout)

        self.attention_layernorms = nn.ModuleList()
        self.attention_layers = nn.ModuleList()
        self.forward_layernorms = nn.ModuleList()
        self.forward_layers = nn.ModuleList()

        for _ in range(n_blocks):
            #

        self.last_layernorm = nn.LayerNorm(hidden_dim, eps=1e-8)

    def encode(self, input_seq: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = input_seq.shape
        positions = torch.arange(seq_len, device=input_seq.device).unsqueeze(0).expand(batch_size, -1)

        x = self.item_embedding(input_seq) * (self.hidden_dim ** 0.5)
        x = x + self.position_embedding(positions)
        x = self.embedding_dropout(x)

        padding_mask = input_seq == 0
        causal_mask = #

        for attn_norm, attn, ffn_norm, ffn in zip(
            self.attention_layernorms, self.attention_layers, self.forward_layernorms, self.forward_layers,
        ):
            #

        return self.last_layernorm(x)

    def forward(self, input_seq: torch.Tensor, target_pos: torch.Tensor, target_neg: torch.Tensor):
        hidden = self.encode(input_seq)
        pos_logits = (hidden * self.item_embedding(target_pos)).sum(-1)
        neg_logits = (hidden * self.item_embedding(target_neg)).sum(-1)
        return pos_logits, neg_logits

    def score_all_items(self, input_seq: torch.Tensor) -> torch.Tensor:
        """Скор каждого айтема каталога для последней позиции последовательности."""
        hidden = self.encode(input_seq)
        last_hidden = hidden[:, -1, :]
        return #

In [ ]:
class SASRec(nn.Module):
    def __init__(self, n_items: int, max_len: int, hidden_dim: int = 64, n_heads: int = 2, n_blocks: int = 2, dropout: float = 0.0):
        super().__init__()
        self.n_items = n_items
        self.max_len = max_len
        self.hidden_dim = hidden_dim

        # index 0 переиспользуется и под padding, и под "нет айтема" — его
        # эмбеддинг зафиксирован в нуле (padding_idx) и никогда не участвует
        # в предсказаниях (см. score_all_items).
        self.item_embedding = nn.Embedding(n_items + 1, hidden_dim, padding_idx=0)
        self.position_embedding = nn.Embedding(max_len, hidden_dim)
        self.embedding_dropout = nn.Dropout(dropout)

        self.attention_layernorms = nn.ModuleList()
        self.attention_layers = nn.ModuleList()
        self.forward_layernorms = nn.ModuleList()
        self.forward_layers = nn.ModuleList()

        for _ in range(n_blocks):
            # TODO: attention-блок в pre-norm стиле: сначала нормализация,
            # затем многоголовое внимание
            #
            # TODO: feed-forward блок, тоже pre-norm: сначала нормализация,
            # затем полносвязная подсеть, расширяющая размерность в 4 раза,
            # с нелинейностью GELU и dropout, и сужающая обратно до hidden_dim
            # с ещё одним dropout

        self.last_layernorm = nn.LayerNorm(hidden_dim, eps=1e-8)

    def encode(self, input_seq: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = input_seq.shape
        positions = torch.arange(seq_len, device=input_seq.device).unsqueeze(0).expand(batch_size, -1)

        x = self.item_embedding(input_seq) * (self.hidden_dim ** 0.5)
        x = x + self.position_embedding(positions)
        x = self.embedding_dropout(x)

        padding_mask = input_seq == 0
        causal_mask = ... # TODO: каузальная маска формы (seq_len, seq_len):
        # True строго выше диагонали — то есть в позициях, куда смотреть запрещено

        for attn_norm, attn, ffn_norm, ffn in zip(
            self.attention_layernorms, self.attention_layers, self.forward_layernorms, self.forward_layers,
        ):
            x = ... # TODO: pre-norm + self-attention + residual
            x = ... # TODO: pre-norm + feed-forward + residual
            x = x * (~padding_mask).unsqueeze(-1)  # не даём паддингу "просочиться" через skip-connection

        return self.last_layernorm(x)

    def forward(self, input_seq: torch.Tensor, target_pos: torch.Tensor, target_neg: torch.Tensor):
        hidden = self.encode(input_seq)
        pos_logits = (hidden * self.item_embedding(target_pos)).sum(-1)
        neg_logits = (hidden * self.item_embedding(target_neg)).sum(-1)
        return pos_logits, neg_logits

    def score_all_items(self, input_seq: torch.Tensor) -> torch.Tensor:
        """Скор каждого айтема каталога для последней позиции последовательности."""
        hidden = self.encode(input_seq)
        last_hidden = hidden[:, -1, :]
        return ... # TODO: скалярное произведение last_hidden со всеми эмбеддингами айтемов

Обёртка в интерфейс `recommend_to_users`, чтобы SASRec можно было гонять через
ту же `evaluate_model`. Инференс идёт чанками (`inference_batch_size`), а не
одним гигантским батчем на всех пользователей сразу: при оценке на val/test
`recommend_to_users` вызывается сразу для тысяч пользователей, и без
чанкования self-attention пытается создать тензоры внимания размера
`(n_users, n_heads, max_len, max_len)` за один проход — это легко упирается в
`CUDA out of memory` даже на неплохой видеокарте. Чанки фиксированного размера
решают это независимо от того, сколько пользователей мы оцениваем.


In [ ]:
class SASRecRecommender:
    def __init__(
        self, model: SASRec, user_sequences: Dict, id2item: np.ndarray, max_len: int,
        device=DEVICE, inference_batch_size: int = 512,
    ):
        self.model = model.eval()
        self.user_sequences = user_sequences  # уже со сдвигом +1, см. build_user_sequences
        self.id2item = id2item
        self.max_len = max_len
        self.device = device
        self.inference_batch_size = inference_batch_size

    def recommend_to_users(self, user_ids: Sequence, topn: int = 10) -> Dict:
        recommendations = {uid: [] for uid in user_ids}
        active_users = [uid for uid in user_ids if self.user_sequences.get(uid)]

        for start in range(0, len(active_users), self.inference_batch_size):
            batch_users = active_users[start:start + self.inference_batch_size]
            batch_inputs, batch_seen = [], []
            for uid in batch_users:
                full_history = self.user_sequences[uid]
                # На вход модели идёт только последнее окно длиной max_len (self-attention
                # физически не видит ничего раньше), но маскировать из рекомендаций нужно
                # ВСЮ историю пользователя, а не только это окно — иначе айтемы за пределами
                # окна (пользователь их уже смотрел, просто давно) могут "вернуться" в топ.
                trimmed = full_history[-self.max_len:]
                padded = [0] * (self.max_len - len(trimmed)) + trimmed
                batch_inputs.append(padded)
                batch_seen.append(set(full_history))

            input_tensor = torch.tensor(batch_inputs, dtype=torch.long, device=self.device)
            with torch.no_grad():
                scores = self.model.score_all_items(input_tensor).cpu().numpy()
            scores[:, 0] = -np.inf  # индекс 0 — паддинг, не настоящий айтем

            for row_scores, uid, seen in zip(scores, batch_users, batch_seen):
                row_scores = row_scores.copy()
                for shifted_idx in seen:
                    row_scores[shifted_idx] = -np.inf
                top = np.argpartition(-row_scores, topn)[:topn]
                top = top[np.argsort(-row_scores[top])]
                recommendations[uid] = self.id2item[top - 1].tolist()  # снимаем сдвиг на паддинг

        return recommendations

In [ ]:
def sasrec_bce_loss(pos_logits: torch.Tensor, neg_logits: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    pos_loss = ... # TODO: BCE_with_logits
    neg_loss = ... # TODO: BCE_with_logits
    loss = (pos_loss + neg_loss) * mask
    return loss.sum() / mask.sum().clamp(min=1.0)


def train_sasrec(
    sequences: Dict,
    n_items: int,
    config: SASRecConfig,
    id2item: np.ndarray,
    val_ground_truth: Optional[Dict] = None,
    device=DEVICE,
) -> Tuple[SASRec, List[float]]:
    dataset = SASRecDataset(sequences, n_items=n_items, max_len=config.max_len)
    loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True, drop_last=True)

    model = SASRec(
        n_items=n_items, max_len=config.max_len, hidden_dim=config.hidden_dim,
        n_heads=config.n_heads, n_blocks=config.n_blocks, dropout=config.dropout,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)

    history = []
    best_hitrate, best_state, epochs_without_improve = -1.0, None, 0

    progress = tqdm(range(1, config.n_epochs + 1), desc="SASRec")
    for epoch in progress:
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for input_seq, target_pos, target_neg, mask in loader:
            input_seq = input_seq.to(device)
            target_pos = target_pos.to(device)
            target_neg = target_neg.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()
            pos_logits, neg_logits = model(input_seq, target_pos, target_neg)
            loss = sasrec_bce_loss(pos_logits, neg_logits, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.grad_clip)
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        history.append(epoch_loss / max(n_batches, 1))
        postfix = {"bce_loss": f"{history[-1]:.4f}"}

        is_last_epoch = epoch == config.n_epochs
        stop_early = False
        if val_ground_truth is not None and (epoch % config.eval_every == 0 or is_last_epoch):
            probe = SASRecRecommender(
                model, sequences, id2item=id2item, max_len=config.max_len,
                device=device, inference_batch_size=config.inference_batch_size,
            )
            recs = probe.recommend_to_users(list(val_ground_truth.keys()), topn=10)
            hr = hit_rate_at_k(recs, val_ground_truth, k=10)
            postfix["val_hr@10"] = f"{hr:.4f}"
            if hr > best_hitrate:
                best_hitrate = hr
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                epochs_without_improve = 0
            else:
                epochs_without_improve += 1
                stop_early = epochs_without_improve >= config.patience

        progress.set_postfix(**postfix)
        if stop_early:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history

In [ ]:
sasrec_config = SASRecConfig()

ml_base = BaseRecommender(ml_data["train"])  # переиспользуем маппинг item2id
ml_train_sequences = build_user_sequences(ml_data["train"], ml_base.item2id, timestamp_col=ml_data["timestamp_col"])

sasrec_module, sasrec_history = train_sasrec(
    ml_train_sequences, n_items=ml_base.n_items, config=sasrec_config,
    id2item=ml_base.id2item, val_ground_truth=ml_data["val_truth"],
)

plt.plot(range(1, len(sasrec_history) + 1), sasrec_history)
plt.title("SASRec: BCE loss по эпохам")
plt.xlabel("Эпоха")
plt.ylabel("BCE loss")
plt.show()

In [ ]:
sasrec_model = SASRecRecommender(
    sasrec_module, ml_train_sequences, id2item=ml_base.id2item,
    max_len=sasrec_config.max_len, inference_batch_size=sasrec_config.inference_batch_size,
)
evaluate_model(sasrec_model, ml_data["val_truth"], ml_data["catalog"], model_name="SASRec")

## 10. Итоговое сравнение

До сих пор каждую модель мы обучали на `train` и валидировали на `val` — это
нормально для промежуточных проверок и подбора гиперпараметров. Для финального
сравнения — по всем правилам: дообучаем (с нуля, тем же способом) каждую
модель на `train + val` и меряем один раз на отложенном `test`
(`ml_data["test_truth"]`), к которому никто по дороге не притрагивался. Для SASRec-моделей не передаём `val_ground_truth` в `train_sasrec` —
`val` теперь часть обучающих данных, и использовать его же для early stopping
было бы странно.


In [ ]:
def run_leaderboard(
    factories: Dict[str, "callable"],
    fit_df: pl.DataFrame,
    ground_truth: Dict,
    catalog: Sequence,
    ks: Sequence[int] = (5, 10, 20),
) -> pl.DataFrame:
    """Обучает каждую модель один раз на fit_df и сразу оценивает её через
    evaluate_model."""
    tables = []
    for name, factory in factories.items():
        model = factory(fit_df)
        tables.append(evaluate_model(model, ground_truth, catalog, ks=ks, model_name=name))
    return pl.concat(tables)

In [ ]:
def _fit_sasrec(train_val_df: pl.DataFrame, config: SASRecConfig) -> SASRecRecommender:
    base = BaseRecommender(train_val_df)
    sequences = build_user_sequences(train_val_df, base.item2id, timestamp_col=ml_data["timestamp_col"])
    module, _ = train_sasrec(sequences, n_items=base.n_items, config=config, id2item=base.id2item)
    return SASRecRecommender(
        module, sequences, id2item=base.id2item,
        max_len=config.max_len, inference_batch_size=config.inference_batch_size,
    )


ml_factories = {
    "Random": lambda df: RandomRecommender(df),
    "TopPopular": lambda df: TopPopularRecommender(df),
    "ItemKNN": lambda df: ItemKNNRecommender(df, k_neighbours=100),
    "ALS": lambda df: ALSRecommender(df, factors=64, iterations=15, regularization=0.05, alpha=20.0),
    "EASE": lambda df: EASERecommender(df, reg=100.0),
    "SASRec": lambda df: _fit_sasrec(df, sasrec_config),
}

ml_leaderboard = run_leaderboard(
    ml_factories, ml_data["train_val"], ml_data["test_truth"], ml_data["catalog"], ks=(5, 10, 20),
)
display(ml_leaderboard)

In [ ]:
MODEL_COLORS = {
    "Random": "#2a78d6",
    "TopPopular": "#eb6834",
    "ItemKNN": "#1baf7a",
    "ALS": "#eda100",
    "EASE": "#e87ba4",
    "SASRec": "#008300",
}


def plot_leaderboard(leaderboard: pl.DataFrame, title: str) -> None:
    metrics = ["HitRate", "Recall", "NDCG", "Coverage"]
    ks = sorted(leaderboard["k"].unique().to_list())
    models = [m for m in MODEL_COLORS if m in leaderboard["model"].unique().to_list()]
    colors = [MODEL_COLORS[m] for m in models]

    fig, axes = plt.subplots(
        len(ks), len(metrics), figsize=(4 * len(metrics), 3.2 * len(ks)), sharex=True,
    )
    for row, k in enumerate(ks):
        subset = leaderboard.filter(pl.col("k") == k).to_pandas().set_index("model").loc[models]
        for col, metric in enumerate(metrics):
            ax = axes[row, col]
            ax.bar(models, subset[metric].to_numpy(), color=colors, width=0.6)
            ax.set_title(f"{metric}@{k}")
            if row == len(ks) - 1:
                ax.tick_params(axis="x", rotation=40)
            else:
                ax.tick_params(axis="x", labelbottom=False)

    legend_handles = [plt.Rectangle((0, 0), 1, 1, color=color) for color in colors]
    fig.legend(legend_handles, models, loc="upper center", ncol=len(models), bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(title, y=1.06)
    plt.tight_layout()
    plt.show()


plot_leaderboard(ml_leaderboard, title="MovieLens-1M, test")

### Diversity рекомендаций между моделями

Метрики выше усредняют качество по пользователям, но не отвечают на вопрос
"а рекомендуют ли две модели, по сути, одно и то же?". Если топ-k одной модели
почти всегда совпадает с топ-k другой — например, персонализированная модель
неотличима от TopPopular — это заметно только по самим спискам, а не по
HitRate/NDCG.

Для каждой пары моделей берём одну и ту же случайную выборку из 1000
пользователей, строим их топ-k рекомендации и считаем долю несовпадения
списков (1 − средний коэффициент Жаккара по пользователям):

$$\text{diversity}(A, B) = 1 - \frac{1}{|U|}\sum_{u \in U} \frac{|S_u^A \cap S_u^B|}{|S_u^A \cup S_u^B|}$$

`diversity = 0` — списки полностью совпадают, `diversity = 1` — не пересекаются
вовсе. Диагональ (модель сама с собой) тривиально равна 0 и не показательна.

In [ ]:
def pairwise_recommendation_diversity(
    models: Dict[str, "object"],
    user_ids: Sequence,
    topn: int = 10,
    n_users: int = 1000,
    seed: int = 42,
) -> pl.DataFrame:
    """Для случайной выборки n_users пользователей считает попарную diversity
    топ-n рекомендаций между всеми моделями (1 - средний коэффициент Жаккара
    по пользователям, см. markdown выше)."""
    rng = np.random.default_rng(seed)
    user_ids = list(user_ids)
    sample_size = min(n_users, len(user_ids))
    sampled_users = rng.choice(user_ids, size=sample_size, replace=False).tolist()

    recs_by_model = {
        name: model.recommend_to_users(sampled_users, topn=topn)
        for name, model in models.items()
    }

    names = list(models.keys())
    rows = []
    for name_a in names:
        row = {"model": name_a}
        for name_b in names:
            overlaps = []
            for uid in sampled_users:
                set_a = set(recs_by_model[name_a].get(uid, []))
                set_b = set(recs_by_model[name_b].get(uid, []))
                union = set_a | set_b
                overlaps.append(len(set_a & set_b) / len(union) if union else 0.0)
            row[name_b] = round(1.0 - float(np.mean(overlaps)), 2)
        rows.append(row)
    return pl.DataFrame(rows)


ml_val_models = {
    "Random": random_model,
    "TopPopular": top_popular_model,
    "ItemKNN": itemknn_model,
    "ALS": als_model,
    "EASE": ease_model,
    "SASRec": sasrec_model,
}

ml_diversity = pairwise_recommendation_diversity(
    ml_val_models, list(ml_data["val_truth"].keys()), topn=10, n_users=1000,
)
display(ml_diversity)

## 11. Персональные рекомендации

Добавим свой собственный профиль и посмотрим на
рекомендации. Возьмём EASE: он не требует итеративного обучения, поэтому
переобучить его с добавленным профилем — секундное дело.

Как найти movieId: проще всего найти его по названию в `ml_movies` (см.
пример ниже):

In [ ]:
SELF_USER_ID = -1

# Пример: несколько понравившихся фильмов (можно заменить на свои — найдите
# movieId по названию через ml_movies.filter(pl.col("title").str.contains(...)))
self_movie_titles = [
    "Ghost in the Shell (Kokaku kidotai) (1995)",
    "Perfect Blue (1997)",
    "Akira (1988)",
]
self_movie_ids = [
    ml_movies.filter(pl.col("title") == title)["item_id"][0]
    for title in self_movie_titles
    if ml_movies.filter(pl.col("title") == title).height > 0
]

self_movie_profile = pl.DataFrame({
    "user_id": [SELF_USER_ID] * len(self_movie_ids),
    "item_id": self_movie_ids,
    "rating": [5.0] * len(self_movie_ids),
    "timestamp": [ml_data["train_val"]["timestamp"].max()] * len(self_movie_ids),
})
self_movie_profile = self_movie_profile.select(ml_data["train_val"].columns)  # тот же порядок колонок

ml_train_with_self = pl.concat([ml_data["train_val"], self_movie_profile], how="vertical_relaxed")
ml_ease_personal = EASERecommender(ml_train_with_self, reg=100.0)

recs = ml_ease_personal.recommend_to_users([SELF_USER_ID], topn=10)[SELF_USER_ID]
ml_movies.filter(pl.col("item_id").is_in(recs)).select("item_id", "title", "genres")